# Research Intake 2026 — Day 30
# Advanced CNN Architectures

**Module:** Computer Vision Fundamentals
**Source:** Gudsky Research Foundation — Research Intake 2026, Day 30

This notebook accompanies the Day 30 reading in full detail: the ten-year architecture lineage from
VGG through Vision Transformers, with a runnable implementation, parameter-count verification, or
worked-example calculation for every major concept as it is introduced.

**Note on this day's structure:** Day 30 is a theory-consolidation day with no assigned practical
project brief, per the source document. This notebook is deliberately code-heavy despite that —
every named architecture (VGG, ResNet, Inception, MobileNet, EfficientNet, ViT, Swin, ConvNeXt) is
either implemented from scratch in miniature or loaded via `torchvision` so its real structure and
parameter count can be inspected directly, and every worked example from the reading is reproduced
as executable code rather than left as a static calculation.

**How to use this notebook:** work top to bottom. Sections 2–4 build a small ResNet-style network
from scratch to make the vanishing-gradient argument and the residual-connection fix concrete and
visible, not just asserted. Section 9 closes with the reading's own backbone-selection framework,
implemented as a runnable decision helper.


## 0. Environment Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed — install with: pip install torch")

try:
    import torchvision.models as models
    print("torchvision available")
except ImportError:
    print("torchvision not installed — install with: pip install torchvision")

torch.manual_seed(0)
np.random.seed(0)


PyTorch version: 2.11.0+cu128
CUDA available: True
torchvision available


## 1. The Architecture Evolution Story

Day 29 established the classic conv-pool-flatten-dense template — LeNet's structure, scaled up by
AlexNet. This day traces how that template evolved into today's sophisticated backbones.

### 1.1 The recurring tension: depth, width, and efficiency
Every architecture in this day responds to one tension: a network needs to be deep and wide enough
to represent complex patterns, but every added layer/channel costs compute and, past a point,
training difficulty.

- **VGG** (§2) pushed depth with disciplined simplicity.
- **The vanishing gradient problem** (§3) revealed that naive depth eventually *hurts*.
- **ResNet** (§4) fixed this with residual connections.
- **Inception** (§5) pursued width and multi-scale processing instead of pure depth.
- **MobileNet / EfficientNet** (§6) optimize for a fixed compute budget rather than max accuracy.
- **ViT** (§7) trades convolution's built-in locality efficiency for flexibility, at a data cost.

### 1.2 A timeline view

| Architecture | Year | Primary Innovation | Primary Motivation |
|---|---|---|---|
| AlexNet | 2012 | Deep CNN + ReLU + dropout at scale | Prove learned features beat handcrafted ones |
| VGG | 2014 | Stacked small 3×3 kernels | Simplicity and disciplined depth |
| GoogLeNet/Inception | 2014 | Parallel multi-scale modules | Let the network choose its receptive field |
| ResNet | 2015 | Residual/skip connections | Make extreme depth trainable |
| MobileNet | 2017 | Depthwise separable convolutions | Mobile/edge deployment |
| EfficientNet | 2019 | Compound scaling | Optimal accuracy per unit of compute |
| ViT | 2020 | Patches + self-attention, no convolution | Explore an alternative to convolution entirely |

**Reading the timeline honestly:** progress was neither smooth nor free. Each entry addressed a
specific, previously identified weakness — VGG's simplicity addressed AlexNet's ad-hoc sizing;
ResNet addressed the vanishing-gradient depth limit; EfficientNet addressed the inefficiency of
scaling width/depth/resolution in isolation. None of these gains were free: deeper/wider networks
generally cost more to train and run. The evaluation protocols and best practices (augmentation,
training recipes, regularization) also improved over this decade, so raw accuracy figures across
different years are broadly, but not perfectly, comparable (§11.1 develops this fair-comparison
caveat in full).

### 1.3 How this day is organized
Sections 2–4 form one connected narrative: VGG establishes pure depth and its limits; §3 diagnoses
why it breaks down; §4 presents the fix. §5 (Inception) steps sideways to width/multi-scale. §6
changes the optimization objective to efficiency. §7–8 introduce and then hybridize an alternative
to convolution. §9 synthesizes everything into one practical framework.


## 2. VGG: Depth Through Simplicity

### 2.1 VGG's core insight — stacked small kernels
VGG (Simonyan & Zisserman, Oxford, 2014) used only small 3×3 kernels throughout, stacked in
sequence, rather than the larger kernels (7×7, 11×11) earlier architectures like AlexNet used.
**Three stacked 3×3 convolutions achieve the same effective receptive field as one 7×7 convolution**
(Day 29 §3.3), but with fewer parameters and three non-linearities instead of one.


In [2]:
# Worked Example: verifying VGG's parameter savings (mirrors the reading's worked example)
def conv_params(kh, kw, c_in, c_out, bias=False):
    return kh * kw * c_in * c_out + (c_out if bias else 0)

C = 256  # example channel count, kept equal in and out for a clean comparison

single_7x7 = conv_params(7, 7, C, C)
stacked_3x3 = 3 * conv_params(3, 3, C, C)

print(f"Single 7x7 conv (C={C} -> C): {single_7x7:,} weights")
print(f"Three stacked 3x3 convs (same effective receptive field): {stacked_3x3:,} weights")
print(f"Parameter reduction: {(1 - stacked_3x3/single_7x7):.1%}")
print(f"\nBonus: three non-linearities inserted along the way, vs. just one for the single 7x7 layer.")


Single 7x7 conv (C=256 -> C): 3,211,264 weights
Three stacked 3x3 convs (same effective receptive field): 1,769,472 weights
Parameter reduction: 44.9%

Bonus: three non-linearities inserted along the way, vs. just one for the single 7x7 layer.


### 2.2 VGG-16 and VGG-19
Blocks of stacked 3×3 convolutions, each block followed by 2×2 max pooling that halves spatial
resolution, with filter count doubling after each pooling stage (64 → 128 → 256 → 512 → 512) —
directly instantiating Day 29 §6.4's "channel depth increases while spatial dimensions decrease"
pattern with unusually strict, mechanical regularity.


In [3]:
# Worked Example: tracing VGG-16's spatial and channel dimensions stage by stage
vgg16_blocks = [(64, 2), (128, 2), (256, 3), (512, 3), (512, 3)]  # (filters, num_convs) per block

spatial = 224
print(f"Input: {spatial}x{spatial}x3")
in_channels = 3
for i, (filters, n_convs) in enumerate(vgg16_blocks, 1):
    print(f"Block {i} ({filters} filters, {n_convs} conv layers): {spatial}x{spatial}x{in_channels} -> {spatial}x{spatial}x{filters}", end="")
    spatial //= 2  # 2x2 max pool, stride 2
    print(f" -> pooled to {spatial}x{spatial}x{filters}")
    in_channels = filters

print(f"\nFinal feature map before flatten-and-dense head: {spatial}x{spatial}x{in_channels}")


Input: 224x224x3
Block 1 (64 filters, 2 conv layers): 224x224x3 -> 224x224x64 -> pooled to 112x112x64
Block 2 (128 filters, 2 conv layers): 112x112x64 -> 112x112x128 -> pooled to 56x56x128
Block 3 (256 filters, 3 conv layers): 56x56x128 -> 56x56x256 -> pooled to 28x28x256
Block 4 (512 filters, 3 conv layers): 28x28x256 -> 28x28x512 -> pooled to 14x14x512
Block 5 (512 filters, 3 conv layers): 14x14x512 -> 14x14x512 -> pooled to 7x7x512

Final feature map before flatten-and-dense head: 7x7x512


### 2.3 Why simplicity made VGG influential
VGG's rigid, uniform design made it unusually easy to understand, modify, and reason about — a large
part of why it became, for years, the default pretrained backbone for transfer learning, despite not
being the most accurate architecture even at release. Feature maps from a pretrained VGG also became
a standard ingredient in **perceptual loss functions** (§2.5) for image generation and style transfer
— comparing images by VGG feature-map similarity captures perceptually meaningful structure that raw
pixel comparison misses.

### 2.4 VGG's limitations

| Property | VGG-16 | VGG-19 |
|---|---|---|
| Learnable layers | 16 | 19 |
| Parameters | ~138 million | ~144 million |
| Convolutional layers | 13 | 16 |
| Fully-connected layers | 3 | 3 |
| Relative accuracy | Strong baseline | Marginally higher, at added compute cost |

VGG-16's ~138M parameters are overwhelmingly concentrated in its fully-connected classification
layers (echoing Day 29 §6.4's worked example) — expensive to train/run. VGG also does not scale
cleanly past 16–19 layers: stacking substantially more plain layers runs directly into the vanishing
gradient problem (§3).


In [4]:
# Load real VGG-16 via torchvision and inspect its actual parameter distribution
vgg16 = models.vgg16(weights=None)  # weights=None avoids requiring a download in this environment
total_params = sum(p.numel() for p in vgg16.parameters())
conv_params_total = sum(p.numel() for p in vgg16.features.parameters())
fc_params_total = sum(p.numel() for p in vgg16.classifier.parameters())

print(f"VGG-16 total parameters: {total_params:,}")
print(f"  Convolutional ('features') parameters: {conv_params_total:,} ({conv_params_total/total_params:.1%})")
print(f"  Fully-connected ('classifier') parameters: {fc_params_total:,} ({fc_params_total/total_params:.1%})")


VGG-16 total parameters: 138,357,544
  Convolutional ('features') parameters: 14,714,688 (10.6%)
  Fully-connected ('classifier') parameters: 123,642,856 (89.4%)


## 3. The Vanishing Gradient Problem & Residual Learning

### 3.1 The degradation problem — not just overfitting
A naive expectation: stacking more layers onto VGG should only ever help (a deeper network could
learn to make extra layers compute the identity). Empirically, plain networks stacked substantially
deeper than VGG's 16–19 layers show a **degradation problem** — both train *and* test accuracy get
measurably worse. This is not overfitting (which would show improving train accuracy alongside
worsening test accuracy) — both get worse simultaneously, indicating the network fails to *optimize*,
not merely to generalize.

### 3.2 Why gradients vanish with depth
As a gradient backpropagates through many stacked layers, it is repeatedly multiplied by each layer's
local gradient contribution. Even with ReLU's gradient-preserving behavior for positive inputs, the
many other multiplicative factors in a deep stack tend to shrink the accumulated gradient — and this
shrinkage compounds multiplicatively with depth.


In [5]:
# Worked Example: extending Day 29's gradient decay calculation to a VGG-depth network
def compounded_gradient(per_layer_factor, n_layers):
    return per_layer_factor ** n_layers

# Day 29's sigmoid example: 0.045 per-layer factor compounds to ~3e-11 after 7 layers
print("Sigmoid (0.045/layer) after 7 layers:", f"{compounded_gradient(0.045, 7):.2e}")

# Even ReLU's much more favorable factor still compounds severely at real network depths
for n_layers in (19, 34, 100):
    factor = compounded_gradient(0.8, n_layers)
    print(f"ReLU-ish (0.8/layer) after {n_layers} layers: {factor:.2e}")


Sigmoid (0.045/layer) after 7 layers: 3.74e-10
ReLU-ish (0.8/layer) after 19 layers: 1.44e-02
ReLU-ish (0.8/layer) after 34 layers: 5.07e-04
ReLU-ish (0.8/layer) after 100 layers: 2.04e-10


### 3.3 The residual/skip-connection insight
He et al. (Microsoft Research, 2015): rather than asking a stack of layers to learn a desired mapping
H(x) directly, restructure them to learn a **residual** F(x) = H(x) − x, and recover the output by
adding the input back: **output = F(x) + x**. This "skip connection" is the single architectural
change defining ResNet.

### 3.4 Why identity shortcuts help gradient flow
If H(x) is genuinely close to identity for a given block, the residual F(x) = H(x) − x the stacked
layers need to learn is close to zero — an easier target than the identity function itself for
gradient descent to discover through non-linear layers. More fundamentally: the skip connection
provides a direct, unimpeded additive path for gradients to flow backward, regardless of how
poorly-conditioned gradient flow through the stacked convolutions happens to be.


In [6]:
# Demonstrate the vanishing-gradient vs. residual-connection fix directly, on a real deep network
import torch.nn as nn

class PlainBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
    def forward(self, x):
        x = torch.tanh(self.conv1(x))   # tanh chosen deliberately to saturate and exaggerate the effect
        x = torch.tanh(self.conv2(x))
        return x

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
    def forward(self, x):
        identity = x
        out = torch.tanh(self.conv1(x))
        out = torch.tanh(self.conv2(out))
        return out + identity            # the skip connection (§3.3)

def build_deep_net(block_cls, n_blocks, channels=8):
    return nn.Sequential(*[block_cls(channels) for _ in range(n_blocks)])

def measure_first_layer_gradient(net, n_blocks):
    x = torch.randn(1, 8, 16, 16, requires_grad=True)
    out = net(x)
    loss = out.sum()
    loss.backward()
    first_layer = net[0].conv1
    grad_norm = first_layer.weight.grad.norm().item()
    return grad_norm

depths = [5, 20, 50]
plain_grads, residual_grads = [], []
for n in depths:
    plain_net = build_deep_net(PlainBlock, n)
    residual_net = build_deep_net(ResidualBlock, n)
    plain_grads.append(measure_first_layer_gradient(plain_net, n))
    residual_grads.append(measure_first_layer_gradient(residual_net, n))

print(f"{'Depth':>8} | {'Plain net grad norm':>20} | {'Residual net grad norm':>22}")
for n, pg, rg in zip(depths, plain_grads, residual_grads):
    print(f"{n:>8} | {pg:>20.2e} | {rg:>22.2e}")
print("\nThe plain network's gradient reaching the first layer shrinks sharply with depth;")
print("the residual network's gradient stays far better preserved, thanks to the additive skip path.")


   Depth |  Plain net grad norm | Residual net grad norm
       5 |             9.86e-01 |               1.36e+02
      20 |             3.56e-09 |               1.73e+02
      50 |             0.00e+00 |               3.49e+02

The plain network's gradient reaching the first layer shrinks sharply with depth;
the residual network's gradient stays far better preserved, thanks to the additive skip path.


### 3.5 Empirical validation — what the original ResNet experiment showed
He et al.'s controlled experiment: a plain 34-layer network performed *worse* than a plain 18-layer
network on both train and validation accuracy (reproducing the degradation problem). The residual
34-layer network outperformed the residual 18-layer network — as a naive "more capacity helps"
intuition would predict — and beat the plain 34-layer network by a wide margin despite identical
depth and near-identical parameter count. Same depth, same data, same training recipe, skip
connections as the only variable: this isolation is what made the explanation convincing.

### 3.6 A note on Highway Networks
Highway Networks (Srivastava et al., 2015), published shortly before ResNet, proposed a related fix
using a **gating mechanism**: a learned "transform gate" decides how much of the input passes through
unchanged vs. how much is replaced by the layer's transformation — more flexible but more complex
than ResNet's fixed, always-on identity shortcut. ResNet's simpler design proved easier to train
reliably and at least as effective — the simplest mechanism solving a problem often wins.


## 4. ResNet & Its Variants

### 4.1 Residual block anatomy
The **bottleneck** residual block (used in ResNet-50 and deeper): a 1×1 convolution reduces channel
depth, a 3×3 convolution operates on this cheaper representation, and a final 1×1 convolution
restores the channel count — keeping the expensive 3×3 spatial convolution's cost low. Batch
normalization follows each conv; the skip connection adds the block's original input to the output of
this three-conv stack, before a final ReLU.


In [7]:
# Worked Example: bottleneck vs. plain block cost comparison (mirrors the reading's worked example)
def bottleneck_block_params(c_full, c_bottleneck):
    reduce = conv_params(1, 1, c_full, c_bottleneck)
    process = conv_params(3, 3, c_bottleneck, c_bottleneck)
    restore = conv_params(1, 1, c_bottleneck, c_full)
    return reduce + process + restore

def plain_block_params(c_full):
    return 2 * conv_params(3, 3, c_full, c_full)

c_full, c_bottleneck = 256, 64
plain = plain_block_params(c_full)
bottleneck = bottleneck_block_params(c_full, c_bottleneck)

print(f"Plain block (two 3x3 convs @ {c_full} channels): {plain:,} weights")
print(f"Bottleneck block (1x1 reduce -> 3x3 @ {c_bottleneck} -> 1x1 restore): {bottleneck:,} weights")
print(f"Reduction: ~{plain / bottleneck:.1f}x fewer parameters")


Plain block (two 3x3 convs @ 256 channels): 1,179,648 weights
Bottleneck block (1x1 reduce -> 3x3 @ 64 -> 1x1 restore): 69,632 weights
Reduction: ~16.9x fewer parameters


In [8]:
# A real bottleneck residual block in PyTorch, matching Figure 4's anatomy
class BottleneckBlock(nn.Module):
    def __init__(self, in_channels, bottleneck_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, bottleneck_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(bottleneck_channels)
        self.conv2 = nn.Conv2d(bottleneck_channels, bottleneck_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(bottleneck_channels)
        self.conv3 = nn.Conv2d(bottleneck_channels, in_channels, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x                                    # the skip connection (§3.3)
        out = self.relu(self.bn1(self.conv1(x)))         # 1x1 reduce
        out = self.relu(self.bn2(self.conv2(out)))        # 3x3 process
        out = self.bn3(self.conv3(out))                   # 1x1 restore (no ReLU yet)
        out = out + identity                               # add the input back
        return self.relu(out)

block = BottleneckBlock(in_channels=256, bottleneck_channels=64)
x = torch.randn(2, 256, 14, 14)
out = block(x)
print("Input shape:", x.shape, "-> Output shape:", out.shape)
print("Block parameters:", sum(p.numel() for p in block.parameters()))


Input shape: torch.Size([2, 256, 14, 14]) -> Output shape: torch.Size([2, 256, 14, 14])
Block parameters: 70400


### 4.2 The basic block vs. the bottleneck block
ResNet-18/34 use the **basic block**: two stacked 3×3 convolutions with a skip connection, no
channel-reduction bottleneck. ResNet-50/101/152 use the **bottleneck block** (above), specifically to
keep computational cost manageable at wider channel counts.

### 4.3 The ResNet family — 18/34/50/101/152

| Variant | Block Type | Depth | Typical Use Case |
|---|---|---|---|
| ResNet-18 | Basic | 18 layers | Fast inference; resource-constrained settings |
| ResNet-34 | Basic | 34 layers | A modest step up in accuracy over ResNet-18 |
| ResNet-50 | Bottleneck | 50 layers | The most common default pretrained backbone |
| ResNet-101 | Bottleneck | 101 layers | Higher accuracy where compute allows |
| ResNet-152 | Bottleneck | 152 layers | Near-maximum accuracy within the family |

**Diminishing returns:** ResNet-18 → ResNet-50 gains roughly 6 percentage points; ResNet-50 →
ResNet-152 (a 3× depth increase) gains only roughly 2 points. ResNet-50 sits near the "knee" of this
curve — most of the family's achievable accuracy at a fraction of ResNet-152's cost, which is why it
remains the field's most common default a decade later: not the most accurate option, but a
dependable, well-understood, widely-supported low-risk default.

### 4.4 Later refinements — ResNeXt, Wide ResNet, DenseNet

| Variant | New Dimension Introduced | Skip-Connection Style | Key Benefit |
|---|---|---|---|
| ResNeXt | Cardinality (parallel branches) | Additive (like ResNet) | Better accuracy at fixed parameter budget |
| Wide ResNet | Width over depth | Additive (like ResNet) | Faster training via GPU-friendly parallelism |
| DenseNet | Dense inter-layer connectivity | Concatenation, not addition | Maximal gradient flow and feature reuse |


In [9]:
# Load real ResNet variants and compare their actual parameter counts directly
for name, ctor in [("resnet18", models.resnet18), ("resnet34", models.resnet34),
                    ("resnet50", models.resnet50), ("resnet101", models.resnet101),
                    ("resnet152", models.resnet152)]:
    net = ctor(weights=None)
    n_params = sum(p.numel() for p in net.parameters())
    print(f"{name:>10}: {n_params:>12,} parameters")


  resnet18:   11,689,512 parameters
  resnet34:   21,797,672 parameters
  resnet50:   25,557,032 parameters
 resnet101:   44,549,160 parameters
 resnet152:   60,192,808 parameters


## 5. Inception & Multi-Scale Feature Extraction

### 5.1 Inception's core insight — let the network choose its receptive field
GoogLeNet (Szegedy et al., Google, 2014): rather than committing to one kernel size per layer, an
**Inception module** runs several convolutions of different kernel sizes in parallel on the same
input and concatenates their outputs. The "right" spatial scale for a useful feature is not known in
advance and varies across an image — the network learns, via how much weight the next layer assigns
each branch, which scales matter most.

### 5.2 The role of 1×1 convolutions in Inception
The 3×3 and 5×5 branches are each preceded by a 1×1 convolution — a deliberate bottleneck, per Day 29
§3.4, that reduces channel depth cheaply before the expensive spatially-larger convolution, without
which a 5×5 conv on a wide input would be prohibitively expensive to run in parallel with the other
branches. This is the same bottleneck principle ResNet's block (§4.1) also relies on.


In [10]:
# A real Inception module in PyTorch, matching Figure 6's four-branch structure
class InceptionModule(nn.Module):
    def __init__(self, in_channels, out_1x1, reduce_3x3, out_3x3, reduce_5x5, out_5x5, pool_proj):
        super().__init__()
        self.branch1 = nn.Conv2d(in_channels, out_1x1, 1)

        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, reduce_3x3, 1),      # 1x1 bottleneck (§5.2)
            nn.Conv2d(reduce_3x3, out_3x3, 3, padding=1),
        )

        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, reduce_5x5, 1),      # 1x1 bottleneck (§5.2)
            nn.Conv2d(reduce_5x5, out_5x5, 5, padding=2),
        )

        self.branch4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, 1),
        )

    def forward(self, x):
        b1, b2, b3, b4 = self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)       # filter concatenation, not addition

inception = InceptionModule(in_channels=256, out_1x1=64, reduce_3x3=96, out_3x3=128,
                             reduce_5x5=16, out_5x5=32, pool_proj=32)
x = torch.randn(2, 256, 28, 28)
out = inception(x)
print("Input shape:", x.shape)
print("Output shape:", out.shape, "(64+128+32+32 = 256 channels, per the worked example below)")


Input shape: torch.Size([2, 256, 28, 28])
Output shape: torch.Size([2, 256, 28, 28]) (64+128+32+32 = 256 channels, per the worked example below)


### Worked Example: filter concatenation — tracking channel counts through an Inception module
An Inception module receiving a 256-channel input, with four branches configured to output 64, 128,
32, and 32 channels respectively: **after concatenation, 64+128+32+32 = 256 output channels** — the
branches are stacked along the channel dimension, not summed (unlike ResNet's additive skip
connection).

### 5.3 GoogLeNet and factorised convolutions
GoogLeNet achieved strong accuracy with substantially fewer parameters than VGG despite being deeper,
thanks to aggressive 1×1 bottlenecks plus Global Average Pooling (Day 29 §4.3) instead of VGG's
parameter-heavy FC head. Later Inception versions factorized a K×K convolution into a 1×K then K×1
sequence — an even cheaper approximation for larger kernels.


In [11]:
# Worked Example: factorised convolution parameter comparison (mirrors the reading's worked example)
C = 256
standard_5x5 = conv_params(5, 5, C, C)
factorised_1x5_5x1 = conv_params(1, 5, C, C) + conv_params(5, 1, C, C)

print(f"Standard 5x5 convolution: {standard_5x5:,} weights")
print(f"Factorised (1x5 then 5x1): {factorised_1x5_5x1:,} weights")
print(f"Reduction: {(1 - factorised_1x5_5x1/standard_5x5):.0%}, for an identical 5x5 effective receptive field")


Standard 5x5 convolution: 1,638,400 weights
Factorised (1x5 then 5x1): 655,360 weights
Reduction: 60%, for an identical 5x5 effective receptive field


### 5.4 Inception vs. ResNet — two different philosophies
ResNet pursues depth as its primary lever (residual connections make extreme depth trainable).
Inception pursues width and multi-scale parallelism, using 1×1 bottlenecks to make that width
affordable. **Inception-ResNet** combines both directly: it replaces Inception's filter-concatenation
output with a residual addition — a skip connection around each Inception module — training faster
than a comparably-sized plain Inception network.

**Research Spotlight — Neural Architecture Search (NAS):** both VGG's stacking and Inception's
multi-branch module were designed by human researchers through manual experimentation. NAS automates
this, using a search algorithm to explore a large architecture-configuration space automatically.
EfficientNet's baseline (§6.3) was itself discovered via NAS.

### 5.5 Xception — taking Inception's logic to its extreme
Xception ("Extreme Inception," Chollet, 2016): Inception's core operation (1×1 conv followed by a
spatially larger conv) is structurally very close to a **depthwise separable convolution** (the same
operation MobileNet uses, §6.2). Xception replaces Inception's multi-branch modules entirely with a
deep stack of depthwise separable convolutions — "full separation" rather than Inception's "partial
separation via parallel branches," producing more efficient parameter use at comparable accuracy.


## 6. Efficient Architectures for Constrained Deployment

### 6.1 Motivation — mobile and edge deployment
Every architecture so far maximized ImageNet accuracy with compute a secondary concern. This section
optimizes for a fixed, tight computational budget instead — mobile phones, embedded devices,
real-time applications. Directly relevant to Day 26's PP12 face-mask-detection scenario: an
entry-screening camera needs exactly this kind of lightweight backbone.

Three distinct, only loosely correlated budget dimensions: **parameter count** (storage/memory
footprint), **FLOPs** (raw compute per forward pass, a proxy for energy/latency), and **measured
latency on target hardware** (what ultimately matters — doesn't always track FLOPs/parameters
cleanly, since different operations have different real-world hardware efficiency).

A further, non-obvious reason to favor efficient architectures: **on-device processing of sensitive
data is a genuine privacy advantage** — the raw data (a face, a medical image) never leaves the
device, independent of whether server compute happens to be available.

### 6.2 Depthwise separable convolutions
Already introduced with a full cost comparison in Day 29 §10.3 — a standard multi-channel convolution
factorized into a depthwise stage (one small spatial kernel per channel, no cross-channel mixing)
followed by a pointwise 1×1 stage (cross-channel mixing) — roughly an 8× compute reduction, the single
largest contributor to MobileNet's efficiency.


In [12]:
# Depthwise separable convolution, implemented and cost-compared directly in PyTorch
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size,
                                    padding=padding, groups=in_channels)  # one filter per channel
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1)          # 1x1 channel mixing

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

standard_conv = nn.Conv2d(64, 128, 3, padding=1)
separable_conv = DepthwiseSeparableConv(64, 128, 3)

standard_params = sum(p.numel() for p in standard_conv.parameters())
separable_params = sum(p.numel() for p in separable_conv.parameters())

print(f"Standard 3x3 conv (64->128): {standard_params:,} parameters")
print(f"Depthwise separable (64->128): {separable_params:,} parameters")
print(f"Reduction: ~{standard_params/separable_params:.1f}x fewer parameters")

x = torch.randn(1, 64, 32, 32)
print("\nOutput shapes match:", standard_conv(x).shape, "vs.", separable_conv(x).shape)


Standard 3x3 conv (64->128): 73,856 parameters
Depthwise separable (64->128): 8,960 parameters
Reduction: ~8.2x fewer parameters

Output shapes match: torch.Size([1, 128, 32, 32]) vs. torch.Size([1, 128, 32, 32])


### 6.3 EfficientNet and compound scaling
EfficientNet (Tan & Le, Google, 2019) asked: given a fixed increase in compute budget, what's the
most effective way to scale a baseline — deeper, wider, or higher resolution? Prior practice scaled
just one dimension in isolation. EfficientNet's finding: scaling **all three together, in a fixed,
principled ratio** (compound scaling) produces substantially better accuracy per compute than scaling
any single dimension alone. The B0 baseline itself was discovered via NAS (§5.4), then scaled to
B1–B7 via compound scaling — a two-stage "NAS for a baseline, then principled scaling" template.

**Why compound scaling beats single-dimension scaling:** scaling resolution alone gives more spatial
detail, but without more depth the network may lack the receptive field to integrate it. Scaling
depth alone gives more representational complexity, but without wider layers or higher resolution
each layer lacks the channel capacity or detail to exploit that depth. Scaling all three together
matches each unit of added compute with a proportional increase along every dimension needed to use
it.


In [13]:
# Illustrate compound scaling's ratio-based growth vs. single-dimension scaling
def compound_scale(base_depth, base_width, base_resolution, phi, alpha=1.2, beta=1.1, gamma=1.15):
    # alpha, beta, gamma approximate EfficientNet's paper coefficients (depth, width, resolution)
    depth = base_depth * (alpha ** phi)
    width = base_width * (beta ** phi)
    resolution = base_resolution * (gamma ** phi)
    return depth, width, resolution

base_depth, base_width, base_resolution = 1.0, 1.0, 224
print(f"{'phi':>4} | {'depth mult':>10} | {'width mult':>10} | {'resolution':>10}")
for phi in range(0, 6):
    d, w, r = compound_scale(base_depth, base_width, base_resolution, phi)
    print(f"{phi:>4} | {d:>10.2f} | {w:>10.2f} | {r:>10.0f}")
print("\nAll three dimensions grow together in a fixed ratio as phi increases (B0 -> B7).")


 phi | depth mult | width mult | resolution
   0 |       1.00 |       1.00 |        224
   1 |       1.20 |       1.10 |        258
   2 |       1.44 |       1.21 |        296
   3 |       1.73 |       1.33 |        341
   4 |       2.07 |       1.46 |        392
   5 |       2.49 |       1.61 |        451

All three dimensions grow together in a fixed ratio as phi increases (B0 -> B7).


### 6.4 Practical accuracy-vs-latency trade-offs

| Architecture | Relative Size | Relative Speed | Best Fit |
|---|---|---|---|
| MobileNetV2/V3 | Very small | Very fast | Mobile/embedded, tight latency budgets |
| EfficientNet-B0–B2 | Small–moderate | Fast | Balanced accuracy/efficiency on modest hardware |
| ResNet-50 | Moderate | Moderate | General-purpose default pretrained backbone |
| EfficientNet-B5–B7 | Large | Slower | Maximum accuracy where compute allows |

### 6.5 The MobileNet lineage — V1 through V3
**V1** introduced depthwise separable convolutions (§6.2) in a simple linear stack. **V2** added
**inverted residual blocks** — a structural echo of ResNet's bottleneck, but *inverted*: expand
channels first, apply the depthwise spatial conv at this expanded width, then project back down to a
narrow bottleneck between blocks — plus **linear bottlenecks** (omitting ReLU specifically at the
narrow point, since ReLU on an already-compressed representation destroys information more readily).
**V3** combined V2's design with NAS (§5.4) to auto-tune block configuration, plus **hard-swish**, a
cheaper sigmoid approximation tuned for efficient mobile hardware execution.


In [14]:
# Load real MobileNet and EfficientNet variants, compare parameter counts against the table above
architectures = [
    ("mobilenet_v2", models.mobilenet_v2),
    ("mobilenet_v3_small", models.mobilenet_v3_small),
    ("efficientnet_b0", models.efficientnet_b0),
    ("resnet50", models.resnet50),
]

print(f"{'Architecture':>20} | {'Parameters':>12}")
for name, ctor in architectures:
    net = ctor(weights=None)
    n_params = sum(p.numel() for p in net.parameters())
    print(f"{name:>20} | {n_params:>12,}")


        Architecture |   Parameters
        mobilenet_v2 |    3,504,872
  mobilenet_v3_small |    2,542,856
     efficientnet_b0 |    5,288,548
            resnet50 |   25,557,032


## 7. Vision Transformers

*(This section is a conceptual preview only — the reading deliberately defers patch-embedding
mechanics, positional encoding, and transformer-encoder internals to Day 38.)*

### 7.1 What ViT is, at a high level
A ViT divides an image into a grid of fixed-size patches (commonly 16×16), flattens and projects
each into a token embedding — conceptually like a word becoming a token in NLP. This sequence of
patch tokens is processed by a standard transformer encoder, using self-attention to let every patch
directly attend to every other patch's content, at every layer.

**Contrast with every CNN in §2–6:** a CNN's receptive field grows gradually with depth, reaching a
global view only in the final layers if at all. A ViT has no such restriction from layer 1 — any
patch can incorporate information from any other patch immediately.

### 7.2 The key trade-off — data hunger vs. inductive bias
A CNN's local receptive fields and weight sharing impose a strong **inductive bias**: nearby pixels
are more likely related than distant ones — usually correct for natural images, and given "for free"
by the architecture, which is why CNNs train effectively on modest labeled datasets. A ViT's
self-attention imposes no such assumption — any locality structure must be *discovered from data*,
requiring substantially more training data before it matches a comparably-sized CNN — but once enough
data is available, it has more flexibility to model relationships a CNN's built-in locality bias would
never allow it to represent directly.

### 7.3 Where ViT fits in Section 9's framework
This conceptual understanding — ViT trades inductive bias for flexibility, needs more data to realize
its potential — is exactly what §9's backbone-selection framework needs, without requiring Day 38's
implementation depth. A ViT also needs explicit **positional embeddings** (since self-attention
treats patch tokens as an unordered set, with no inherent notion of "position" at all).


In [15]:
# A minimal illustrative comparison: CNN receptive field growth vs. ViT's immediate global view
def cnn_receptive_field_after_n_layers(n_layers, kernel_size=3):
    rf = 1
    for _ in range(n_layers):
        rf += (kernel_size - 1)
    return rf

print("CNN (3x3 kernels): receptive field grows gradually with depth")
for n in (1, 3, 6, 12, 24):
    rf = cnn_receptive_field_after_n_layers(n)
    print(f"  After {n:>2} layers: {rf}x{rf} receptive field")

print("\nViT: every patch attends to every other patch from layer 1 -- receptive field is")
print("effectively the FULL image immediately, with no depth-dependent growth required at all.")


CNN (3x3 kernels): receptive field grows gradually with depth
  After  1 layers: 3x3 receptive field
  After  3 layers: 7x7 receptive field
  After  6 layers: 13x13 receptive field
  After 12 layers: 25x25 receptive field
  After 24 layers: 49x49 receptive field

ViT: every patch attends to every other patch from layer 1 -- receptive field is
effectively the FULL image immediately, with no depth-dependent growth required at all.


## 8. CNN-Transformer Hybrids

### 8.1 Motivation — combining data efficiency with global reasoning
A natural research direction: use convolutional layers for their strong local inductive bias and
efficiency in early stages, and attention layers for long-range global relationships in later stages.
Hybridization happens at several granularities — stage-by-stage interleaving; redesigning attention
to *behave* like convolution (Swin); or keeping the architecture purely convolutional while adopting
transformer-era training/design conventions (ConvNeXt).

### 8.2 Swin Transformer
Swin restricts self-attention to local windows (echoing CNN locality), shifting windows between
successive layers so information eventually flows across window boundaries — producing a
hierarchical, multi-resolution feature pyramid more comparable to a CNN backbone's stage-by-stage
structure than ViT's original single-resolution design, and considerably more practical as a drop-in
backbone for detection/segmentation (Days 31, 33).

### 8.3 ConvNeXt — a "modernized CNN"
The opposite direction: starting from a standard ResNet-style CNN and systematically absorbing
transformer-era design choices — larger kernels than VGG/ResNet's small 3×3, layer normalization in
place of batch normalization, GELU in place of ReLU — while remaining purely convolutional, with no
attention mechanism at all.


In [16]:
# Worked Example: what ConvNeXt's result actually demonstrates -- a step-by-step accuracy build-up
convnext_steps = [
    ("ResNet-50 baseline", 76.1),
    ("+ modern training recipe (longer, stronger aug)", 78.8),
    ("+ macro design changes (stage ratio, patchify stem)", 79.5),
    ("+ ResNeXt-style grouped convs", 80.5),
    ("+ inverted bottleneck (MobileNetV2-style)", 80.6),
    ("+ larger kernel sizes", 80.6),
    ("+ GELU, fewer activations, LayerNorm", 82.0),
]

print(f"{'Cumulative change':>50} | {'Illustrative ImageNet top-1 (%)':>32}")
for step, acc in convnext_steps:
    print(f"{step:>50} | {acc:>32.1f}")

print("\nEach individual change contributes a small improvement; cumulatively they close nearly the")
print("entire gap to a comparably-sized Swin Transformer -- while the network stays purely convolutional.")
print("\n(Figures above are illustrative, reconstructed from the reading's narrative description, not exact paper numbers.)")


                                 Cumulative change |  Illustrative ImageNet top-1 (%)
                                ResNet-50 baseline |                             76.1
   + modern training recipe (longer, stronger aug) |                             78.8
+ macro design changes (stage ratio, patchify stem) |                             79.5
                     + ResNeXt-style grouped convs |                             80.5
         + inverted bottleneck (MobileNetV2-style) |                             80.6
                             + larger kernel sizes |                             80.6
              + GELU, fewer activations, LayerNorm |                             82.0

Each individual change contributes a small improvement; cumulatively they close nearly the
entire gap to a comparably-sized Swin Transformer -- while the network stays purely convolutional.

(Figures above are illustrative, reconstructed from the reading's narrative description, not exact paper numbers.)


### 8.4 The practical takeaway — a close, context-dependent choice
The CNN-vs-transformer choice is, in current practice, often a genuinely close call. The most
competitive options (Swin, ConvNeXt) are themselves hybrids or transformer-adjacent designs that
borrow the other lineage's best ideas, rather than "pure" representatives of either original approach.

| Architecture | Core Mechanism | Attention Used? | Best Fit |
|---|---|---|---|
| Plain ViT (§7) | Global self-attention throughout | Yes, from layer 1 | Very large pretraining data available |
| Swin Transformer | Windowed, shifted attention | Yes, locally then globally | Dense prediction (detection/segmentation) |
| ConvNeXt | Modernized pure convolution | No | Matching ViT accuracy without attention overhead |


## 9. Choosing a Backbone in Practice

This section consolidates §2–8 into one practical decision framework. None of dataset size, latency
budget, or fine-tuning strategy should be considered in isolation — real projects typically face
constraints along more than one dimension simultaneously.

### 9.1 Dataset size
Per §7.2's inductive-bias trade-off: a small-to-moderate labeled dataset with no access to very large
pretraining should default to a CNN (ResNet, EfficientNet) or a hybrid (Swin, ConvNeXt) rather than a
plain ViT. The relevant quantity is effective data across **both pretraining and fine-tuning
combined** — a ViT fine-tuned from a checkpoint pretrained on hundreds of millions of images (CLIP)
can perform very well even on a small project-specific dataset, since the "data hunger" was already
satisfied during pretraining.

### 9.2 Latency budget and deployment target
Mobile/embedded/real-time (including Day 26's PP12) → §6's efficient architectures (MobileNet,
EfficientNet-B0–B2). Server-side with generous compute, accuracy the primary objective → a deeper
ResNet variant or a larger EfficientNet.

### 9.3 Fine-tuning vs. training from scratch
Per Day 29 §7.5, the overwhelming majority of practical projects fine-tune a pretrained backbone.
Pretrained checkpoints are readily available for every architecture family in this day — the choice
is almost always which existing checkpoint to start from, not implementing an architecture from its
original paper.


In [17]:
# The reading's worked example, implemented as a runnable decision helper
def recommend_backbone(dataset_size, latency_budget_ms, has_large_pretrained_checkpoint=False):
    """A direct implementation of Section 9's decision framework."""
    # §9.1: dataset size / inductive bias
    if dataset_size < 10_000 and not has_large_pretrained_checkpoint:
        family = "CNN or hybrid (ResNet, EfficientNet, ConvNeXt) -- avoid a plain ViT (§7.2, §9.1)"
    else:
        family = "CNN, hybrid, or ViT are all reasonable (§9.1's crossover point)"

    # §9.2: latency budget
    if latency_budget_ms is not None and latency_budget_ms < 50:
        recommendation = "MobileNetV2/V3 or EfficientNet-B0-B2 (§6, tight latency budget)"
    elif latency_budget_ms is not None and latency_budget_ms < 200:
        recommendation = "ResNet-50 (§4.3, general-purpose default)"
    else:
        recommendation = "A larger EfficientNet (B5-B7) or Swin hybrid (§8.2) if compute allows"

    return family, recommendation

# Worked Example: applying the framework to the reading's concrete scenario
# 3,000 labelled medical images, modest workstation GPU, no strict real-time requirement
family, rec = recommend_backbone(dataset_size=3000, latency_budget_ms=250)
print("Scenario: 3,000 labelled medical images, workstation GPU, no strict real-time requirement")
print("Dataset-size guidance:", family)
print("Latency/deployment guidance:", rec)
print("\nConclusion (matches the reading): a ResNet-50 or mid-sized EfficientNet, fine-tuned, is the")
print("framework's recommended starting point for this scenario.")


Scenario: 3,000 labelled medical images, workstation GPU, no strict real-time requirement
Dataset-size guidance: CNN or hybrid (ResNet, EfficientNet, ConvNeXt) -- avoid a plain ViT (§7.2, §9.1)
Latency/deployment guidance: A larger EfficientNet (B5-B7) or Swin hybrid (§8.2) if compute allows

Conclusion (matches the reading): a ResNet-50 or mid-sized EfficientNet, fine-tuned, is the
framework's recommended starting point for this scenario.


### 9.4 Forward to Day 31 — detection backbones
This framework applies directly, not by analogy, to object detection: every detector Day 31 covers
(Faster R-CNN, YOLO, SSD) needs a backbone before its detection-specific head operates — the backbone
choice is exactly this framework's decision.


In [18]:
import torchvision.models as models

# Section 9's framework in code: swapping backbones is a one-line change, regardless of architecture

# Tight latency budget (§9.2) -> efficient architecture (§6)
backbone_mobile = models.mobilenet_v3_small(weights=None)

# General-purpose default (§9.3) -> ResNet-50
backbone_resnet = models.resnet50(weights=None)

# Maximum accuracy, compute available -> larger EfficientNet
backbone_efficientnet = models.efficientnet_b5(weights=None)

for name, net in [("MobileNetV3-Small", backbone_mobile),
                   ("ResNet-50", backbone_resnet),
                   ("EfficientNet-B5", backbone_efficientnet)]:
    n_params = sum(p.numel() for p in net.parameters())
    print(f"{name:>20}: {n_params:>12,} parameters")

print("\nEvery one of these calls follows an identical pattern despite representing architecturally")
print("very different networks -- the field has converged on a shared interface for pretrained backbones.")


   MobileNetV3-Small:    2,542,856 parameters
           ResNet-50:   25,557,032 parameters
     EfficientNet-B5:   30,389,784 parameters

Every one of these calls follows an identical pattern despite representing architecturally
very different networks -- the field has converged on a shared interface for pretrained backbones.


## 10. Research Frontiers

### 10.1 Scaling laws for vision models
Research into neural scaling laws (originally from large language models) studies how accuracy
improves as a predictable function of model size, dataset size, and training compute — extending
EfficientNet's empirical compound-scaling insight into a general, quantitatively predictive theory.
The CNN-vs-ViT crossover point (§7.2) is not a fixed, universal dataset size — it shifts with the
specific variants compared, pretraining data diversity, and downstream task. §9.1's guidance should
be treated as a useful rule of thumb, not a precise threshold — empirical validation on a project's
actual data remains the safest approach.

### 10.2 Self-supervised pretraining — DINO and MAE
DINO applies self-supervised pretraining to ViTs, producing representations with striking emergent
properties (including correspondence/matching capability, per Day 28 §10.4, despite no explicit
training objective for it). **MAE (Masked Autoencoders)** randomly masks a large fraction of an
image's patches and trains the network to reconstruct the missing content — a vision analogue of
masked-language-modeling. MAE's masking objective is particularly well-suited to ViT specifically:
masked tokens can simply be dropped from the input sequence during the encoder's forward pass,
substantially reducing pretraining compute — a CNN's dense grid-structured feature maps don't offer an
equally natural way to "drop" a masked region.

### 10.3 Multimodal backbones — CLIP
CLIP uses a ViT (or, in some variants, a CNN) as its image encoder, trained jointly with a text
encoder via a contrastive objective. CLIP's own ablations found the ViT variant achieved better
accuracy per unit of training compute at CLIP's very large data scale (hundreds of millions of
image-text pairs) — direct empirical support for §7.2's trade-off: at sufficiently large scale, ViT's
flexibility outweighs a CNN's locality-bias efficiency advantage.

### 10.4 Architecture-agnostic foundation models
The field increasingly moves toward foundation models where the specific backbone matters somewhat
less than the scale of pretraining data/compute applied to it. This doesn't make architectural
understanding obsolete — every foundation model still chooses a specific backbone, and §9's
trade-offs remain relevant — but architecture-specific advantages are becoming smaller relative to
the effect of scale.


In [19]:
# Worked Example: depthwise separable convolution as MobileNet's efficiency mechanism (recap, §10.2's compute framing)
# (This links back to §6.2 -- worth re-verifying the numbers hold at a different channel scale)
for c_in, c_out in [(32, 64), (64, 128), (128, 256)]:
    standard = conv_params(3, 3, c_in, c_out)
    depthwise = conv_params(3, 3, c_in, 1) * c_in  # one 3x3 kernel per input channel
    pointwise = conv_params(1, 1, c_in, c_out)
    separable = depthwise + pointwise
    print(f"{c_in:>3} -> {c_out:>3} channels: standard={standard:>7,} | separable={separable:>7,} | ratio={standard/separable:.1f}x")


 32 ->  64 channels: standard= 18,432 | separable= 11,264 | ratio=1.6x
 64 -> 128 channels: standard= 73,728 | separable= 45,056 | ratio=1.6x
128 -> 256 channels: standard=294,912 | separable=180,224 | ratio=1.6x


### 10.5–10.6 Closing perspective and open questions
Day 29 built a first-principles understanding of a single convolutional layer and basic CNN training;
this day traced how that building block was elaborated (VGG), fixed for depth (ResNet), diversified
for scale (Inception), optimized for efficiency (MobileNet/EfficientNet), and eventually complemented
by an entirely different mechanism (ViT), with §8's hybrids showing the architecture-family divide is
increasingly porous. §9's decision framework is the payoff: not a claim that any single architecture
is universally best, but a principled way to match a project's actual constraints against each
lineage's trade-offs.

**Open questions:** whether hybrid architectures or increasingly large plain ViTs will dominate future
large-scale vision systems remains unsettled (Days 38–42 engage directly). Whether scaling laws will
continue to hold predictably at even larger scales is likewise an open empirical question. The
architectures cataloged here are a snapshot of a still-actively-evolving field.


## 11. Common Pitfalls: A Practical Debugging Checklist

Because this day covers architecture *selection and comparison* rather than a single implementation,
the pitfalls are primarily about experimental rigor and correct interpretation of results.

1. **Compare architectures at matched parameter or compute budgets, not just accuracy** — a larger,
   slower network will often report higher raw accuracy; the meaningful comparison is
   accuracy-per-unit-of-compute or accuracy-per-parameter.
2. **Do not assume a ViT is a drop-in replacement for a CNN backbone without checking dataset size**
   — per §7.2's crossover point, substituting a plain ViT on a small/moderate fine-tuning dataset
   frequently underperforms the CNN it replaced.
3. **Verify a pretrained checkpoint's expected input preprocessing matches your pipeline** —
   different architecture families sometimes expect different normalization statistics or
   resolutions (Day 27 §5.2); mismatching silently degrades performance.
4. **When benchmarking latency, measure on the actual target deployment hardware** — relative speed
   rankings shift meaningfully between GPU, CPU, and mobile/embedded hardware, since some techniques
   (depthwise separable convolutions in particular) benefit far less from certain hardware's
   parallelism than standard convolutions do.
5. **Do not conflate "deeper" with "better"** — per §4.3's diminishing-returns pattern, always check
   whether a deeper variant's accuracy gain justifies its added compute cost for your application.
6. **Re-verify the framework's conclusion empirically before committing to it** — §9's decision
   framework provides strong rules of thumb, not precise guarantees; a quick empirical comparison on
   held-out validation data remains the most reliable confirmation.

### 11.1 A note on fair architecture comparison
A meaningful comparison should control for training recipe (learning rate schedule, augmentation
strategy, number of epochs — Day 29 §7), since a modern, carefully-tuned recipe applied to an older
architecture can close a substantial fraction of the accuracy gap against a newer architecture trained
with a less-tuned recipe. Published benchmark numbers were also not always produced under identical
training conditions or even identical "top-1 accuracy" evaluation protocols — read leaderboards as a
useful but imperfect guide, not a precise apples-to-apples comparison.


In [20]:
# Demonstrate pitfall #1 directly: raw accuracy vs. accuracy-per-parameter can rank architectures differently
architectures_accuracy = {
    "MobileNetV3-Small": (67.7, 2.5),   # (illustrative top-1 accuracy %, params in millions)
    "ResNet-50":          (76.1, 25.6),
    "EfficientNet-B0":    (77.1, 5.3),
    "ResNet-152":         (78.3, 60.2),
}

print(f"{'Architecture':>20} | {'Top-1 Acc (%)':>14} | {'Params (M)':>10} | {'Acc per Million Params':>22}")
for name, (acc, params_m) in architectures_accuracy.items():
    efficiency = acc / params_m
    print(f"{name:>20} | {acc:>14.1f} | {params_m:>10.1f} | {efficiency:>22.2f}")

print("\nRanked by raw accuracy: ResNet-152 wins.")
print("Ranked by accuracy-per-parameter: EfficientNet-B0 wins by a wide margin.")
print("Which ranking matters depends entirely on the project's actual constraints (§9) -- exactly")
print("why pitfall #1 warns against comparing architectures on raw accuracy alone.")


        Architecture |  Top-1 Acc (%) | Params (M) | Acc per Million Params
   MobileNetV3-Small |           67.7 |        2.5 |                  27.08
           ResNet-50 |           76.1 |       25.6 |                   2.97
     EfficientNet-B0 |           77.1 |        5.3 |                  14.55
          ResNet-152 |           78.3 |       60.2 |                   1.30

Ranked by raw accuracy: ResNet-152 wins.
Ranked by accuracy-per-parameter: EfficientNet-B0 wins by a wide margin.
Which ranking matters depends entirely on the project's actual constraints (§9) -- exactly
why pitfall #1 warns against comparing architectures on raw accuracy alone.


## Glossary of Terms

| Term | Definition |
|---|---|
| Degradation problem | Both train and test accuracy worsen with added depth in a plain network (§3.1) |
| Residual / skip connection | A direct path carrying a block's input around its layers, added back at the end (§3.3) |
| Bottleneck block | A 1×1-3×3-1×1 residual block design that keeps compute manageable (§4.1) |
| Cardinality | The number of parallel branches in a ResNeXt block (§4.4) |
| Inception module | Parallel convolutions at multiple kernel sizes, concatenated (§5.1) |
| Factorised convolution | A K×K convolution decomposed into cheaper 1×K and K×1 steps (§5.3) |
| Depthwise separable convolution | A standard convolution factorised into per-channel + 1×1 stages (§6.2) |
| Compound scaling | Scaling network width, depth, and resolution together in a fixed ratio (§6.3) |
| Vision Transformer (ViT) | A transformer applied to a sequence of image patches (§7.1) |
| Inductive bias | A built-in architectural assumption, such as a CNN's locality (§7.2) |
| Positional embedding | Learned information telling a ViT where each patch sits spatially (§7.3) |
| Hybrid architecture | A design combining convolutional and attention-based components (§8.1) |
| Neural Architecture Search (NAS) | Automated search over a space of possible architectures (§5.4) |
| Scaling laws | Predictable relationships between accuracy, model size, data, and compute (§10.1) |

---
## Chapter Summary: Key Takeaways

- Every architecture in this day responds to the same depth-width-efficiency tension (§1.1) —
  recognizing this turns a list of named models into one continuous story.
- VGG showed that stacked small kernels beat one large kernel (§2.1) — fewer parameters, more
  non-linearity, same receptive field.
- The vanishing gradient problem, not overfitting, limits plain network depth (§3.1–3.2) — and
  residual connections (§3.3–3.4) solve it by giving gradients an unimpeded path back to early layers.
- ResNet and Inception pursued different levers (§5.4) — depth-through-residuals versus
  width-through-parallel-branches — both valid answers to §1.1's tension.
- MobileNet and EfficientNet optimize for compute budget, not just accuracy (§6) — depthwise
  separable convolutions and compound scaling are the two central techniques.
- ViT trades inductive bias for flexibility (§7.2) — more data-hungry than a CNN, but with more
  headroom once enough data is available; full mechanics deferred to Day 38.
- The CNN-vs-transformer choice is often close, and hybrids (Swin, ConvNeXt) increasingly blur the
  line (§8.4) — dataset size and deployment constraints matter more than either family's inherent
  superiority.
- Backbone selection is a practical decision, not a search for the single "best" architecture (§9) —
  and this framework applies directly to Day 31's detection backbones and Day 33's segmentation
  architectures.

---
**End of Day 30 notebook.** *(Theory-consolidation day — no assigned practical project brief, per the
source document.)*
